# Reproducing Table 5 of the NRP-Exp Paper

This notebook reproduces the illustrative workflow reported in Table 5 of the paper.

Configuration:

- Instance: `Concise_Case_11`
- Variant: `Excl(r7)`
- Objectives:
  - Satisfaction (maximize)
  - Effort (minimize)
- Constraint:
  - Effort ≤ 28
- Solver:
  - Brute Force
- Parameter:
  - max_evaluations = 30000

The notebook executes exactly the workflow reported in the paper and computes the corresponding quality indicators.

In [12]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    current = start.resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "src").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not locate project root."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

C:\Users\imagu\OneDrive\Escritorio\NRPExperimentFramework


## Import Framework Components

Import the components required to reproduce the workflow.

In [13]:
import json
import time
import pandas as pd

from src.domain.core.loader import GenericJSONLoader
from src.domain.core.pipeline import PreprocessingEngine
from src.domain.plugins.nrp import NRPPlugin

from src.solving.algorithms.brute_force import BruteForceSolver

from src.solving.core.problem import (
    OptimizationProblem,
    ConstraintSpec,
    ThresholdOperator,
)

## Load the Original Instance

Load the `Concise_Case_11` instance distributed with NRP-Exp.

In [14]:
INSTANCE_PATH = (
    PROJECT_ROOT
    / "examples"
    / "instances"
    / "Concise_Case_11.json"
)

raw_json = INSTANCE_PATH.read_text(
    encoding="utf-8"
)

base_model = GenericJSONLoader.load(
    raw_json,
    NRPPlugin,
)

print(base_model.name)

Concise_Case_11


## Execute Preprocessing

Apply aggregation, coupling resolution, value-dependency processing, and exclusion branching.

In [15]:
processed_models = PreprocessingEngine.process(
    initial_model=base_model,
    pipeline_options={}
)

print(
    "Generated variants:",
    len(processed_models)
)

Generated variants: 2


## Inspect Generated Variants

The paper uses the variant corresponding to `Excl(r7)`.

In [16]:
for idx, model in enumerate(processed_models):
    print(
        idx,
        getattr(
            model,
            "branch_name",
            f"Variant {idx}"
        ),
        len(model.items)
    )

0 Excl(r7) 6
1 Excl(r9) 9


## Select the Variant Reported in Table 5

In [17]:
SELECTED_VARIANT_INDEX = 0

model = processed_models[
    SELECTED_VARIANT_INDEX
]

print(
    getattr(
        model,
        "branch_name",
        "Variant"
    )
)

Excl(r7)


## Configure Objectives and Constraints

Table 5 uses:

- Satisfaction (max)
- Effort (min)
- Effort ≤ 28

In [18]:
OBJECTIVES = {
    "satisfaction": "max",
    "effort": "min",
}

constraints = {
    "effort": ConstraintSpec(
        attribute="effort",
        operator=ThresholdOperator.LESS_EQUAL,
        value=28.0,
    )
}

problem = OptimizationProblem(
    model=model,
    objectives=OBJECTIVES,
    constraints=constraints,
)

## Execute  Brute Force

Configuration reported in Table 5:

- max_evaluations = 30000

In [19]:
solver = BruteForceSolver(
    max_evaluations=30000
)

start = time.perf_counter()

pareto_front = solver.solve(problem)

elapsed = (
    time.perf_counter() - start
)

pareto_front.set_objective_directions(
    problem.get_objective_directions()
)

print(
    "Execution time:",
    elapsed
)

print(
    "Solutions:",
    len(pareto_front.solutions)
)

Execution time: 0.01598740000190446
Solutions: 16


## Inspect the Returned Front

In [20]:
pareto_df = pareto_front.to_dataframe()

pareto_df.head(20)

,id,selected_ids,selected_ids_str,is_feasible,satisfaction,effort,dissatisfaction,risk
0,BF_1,"[r10, r11, r12, r6, r8, r9]","r10, r11, r12, r6, r8, r9",True,25.428571,27.2,14.857143,15.333333
1,BF_2,"[r10, r11, r6, r8, r9]","r10, r11, r6, r8, r9",True,23.714286,22.2,13.571429,12.000000
2,BF_4,"[r10, r6, r8, r9]","r10, r6, r8, r9",True,22.000000,18.0,12.285714,8.666667
3,BF_5,"[r11, r12, r6, r8, r9]","r11, r12, r6, r8, r9",True,25.142857,25.2,12.142857,14.000000
4,BF_6,"[r11, r6, r8, r9]","r11, r6, r8, r9",True,23.428571,20.2,10.857143,10.666667
5,BF_8,"[r6, r8, r9]","r6, r8, r9",True,21.714286,16.0,9.571429,7.333333
6,BF_12,"[r10, r6, r8]","r10, r6, r8",True,20.000000,15.0,10.142857,5.666667
7,BF_18,"[r10, r11, r6, r9]","r10, r11, r6, r9",True,19.714286,12.2,7.428571,9.333333
8,BF_20,"[r10, r6, r9]","r10, r6, r9",True,18.000000,8.0,6.142857,6.000000
9,BF_21,"[r11, r12, r6, r9]","r11, r12, r6, r9",True,21.142857,15.2,6.000000,11.333333


## Compute Quality Indicators

Compute the indicators reported in Table 5.

In [21]:
metrics = pareto_front.compute_metrics(
    objective_names=[
        "satisfaction",
        "effort",
    ],
    directions={
        "satisfaction": "max",
        "effort": "min",
    }
)

metrics

{'Solutions': 16,
 'Hypervolume': 643.0765714285714,
 'Spacing': 0.3940477675574443,
 'Objective-space extent': 37.235094264657896,
 'Spread': 37.235094264657896}

## Table 5 Summary

Generate a summary directly comparable to the values reported in the paper.

In [22]:
summary = {
    "Instance": "Concise_Case_11",
    "Variant": getattr(
        model,
        "branch_name",
        "Excl(r7)"
    ),
    "Constraint": "Effort <= 28",
    "Solver": "Brute Force",
    "max_evaluations": 30000,
    "Solutions": len(
        pareto_front.solutions
    ),
    **metrics,
}

pd.Series(summary)

Instance                  Concise_Case_11
Variant                          Excl(r7)
Constraint                   Effort <= 28
Solver                        Brute Force
max_evaluations                     30000
Solutions                              16
Hypervolume                    643.076571
Spacing                          0.394048
Objective-space extent          37.235094
Spread                          37.235094
dtype: object

## Reproduction Completed

This notebook reproduces the workflow reported in Table 5:

- Concise_Case_11
- Excl(r7)
- Satisfaction (max)
- Effort (min)
- Effort ≤ 28
- Brute Force
- max_evaluations = 30000

The generated outputs can be compared directly against the values reported in the manuscript.